# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [65]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"

marketing_customer = pd.read_csv(url)

print(marketing_customer.duplicated().sum())


0


In [66]:
marketing_customer.columns

Index(['Unnamed: 0', 'Customer', 'State', 'Customer Lifetime Value',
       'Response', 'Coverage', 'Education', 'Effective To Date',
       'EmploymentStatus', 'Gender', 'Income', 'Location Code',
       'Marital Status', 'Monthly Premium Auto', 'Months Since Last Claim',
       'Months Since Policy Inception', 'Number of Open Complaints',
       'Number of Policies', 'Policy Type', 'Policy', 'Renew Offer Type',
       'Sales Channel', 'Total Claim Amount', 'Vehicle Class', 'Vehicle Size',
       'Vehicle Type'],
      dtype='object')

In [67]:
marketing_customer.columns = marketing_customer.columns.str.strip().str.lower().str.replace(" ", "_")

In [68]:
marketing_customer.columns

Index(['unnamed:_0', 'customer', 'state', 'customer_lifetime_value',
       'response', 'coverage', 'education', 'effective_to_date',
       'employmentstatus', 'gender', 'income', 'location_code',
       'marital_status', 'monthly_premium_auto', 'months_since_last_claim',
       'months_since_policy_inception', 'number_of_open_complaints',
       'number_of_policies', 'policy_type', 'policy', 'renew_offer_type',
       'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size',
       'vehicle_type'],
      dtype='object')

In [69]:
marketing_customer["customer"].duplicated().sum()

np.int64(1776)

In [70]:
#1. Create a new DataFrame that only includes customers who:
#   - have a **low total_claim_amount** (e.g., below $1,000),
#   - have a response "Yes" to the last marketing campaign.
marketing_cust_low_claim=marketing_customer[marketing_customer["response"]=="Yes"]
marketing_cust_low_claim=marketing_cust_low_claim.groupby(["customer","response"])["total_claim_amount"].sum()

In [71]:
type(marketing_cust_low_claim)

pandas.core.series.Series

In [72]:
marketing_cust_low_claim[marketing_cust_low_claim < 1000]

customer  response
AA16582   Yes         511.200000
AA35519   Yes         513.600000
AA56476   Yes         681.600000
AB21519   Yes         281.110788
AB96670   Yes         425.266308
                         ...    
ZV43428   Yes         395.729716
ZX73673   Yes         471.443077
ZX93551   Yes         374.400000
ZZ22047   Yes         331.200000
ZZ49347   Yes         624.000000
Name: total_claim_amount, Length: 1193, dtype: float64

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [79]:
marketing_customer.columns

Index(['unnamed:_0', 'customer', 'state', 'customer_lifetime_value',
       'response', 'coverage', 'education', 'effective_to_date',
       'employmentstatus', 'gender', 'income', 'location_code',
       'marital_status', 'monthly_premium_auto', 'months_since_last_claim',
       'months_since_policy_inception', 'number_of_open_complaints',
       'number_of_policies', 'policy_type', 'policy', 'renew_offer_type',
       'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size',
       'vehicle_type'],
      dtype='object')

In [75]:
yes_df = marketing_customer[marketing_customer["response"] == "Yes"]

In [80]:
premium_analysis = yes_df.groupby(["policy_type", "gender"])[
    "monthly_premium_auto"
].mean().reset_index()

In [81]:
clv_analysis = yes_df.groupby(["policy_type", "gender"])[
    "customer_lifetime_value"
].mean().reset_index()

In [82]:
claim_analysis = yes_df.groupby(["policy_type", "gender"])[
    "total_claim_amount"
].mean().reset_index()

In [ ]:
summary = premium_analysis.merge(clv_analysis, on=["policy_type", "gender"]).merge(
    claim_analysis,on=["policy_type", "gender"])

summary.columns = [
    "policy_type", "gender",
    "Avg Monthly Premium",
    "Avg CLV",
    "Avg Claim Amount"
]

summary

,policy_type,gender,Avg Monthly Premium,Avg CLV,Avg Claim Amount
0,Corporate Auto,F,94.301775,7712.628736,433.738499
1,Corporate Auto,M,92.188312,7944.465414,408.582459
2,Personal Auto,F,98.998148,8339.791842,452.965929
3,Personal Auto,M,91.085821,7448.383281,457.010178
4,Special Auto,F,92.314286,7691.584111,453.280164
5,Special Auto,M,86.343750,8247.088702,429.527942


Best segment is Personal Auto and in that female domain
CLV: 8339 (highest)
Premium: ~99 (highest)
Claims: 453 (moderate)
This is your most valuable customer group

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [93]:
marketing_customer_in_state = marketing_customer.groupby("state").size()
marketing_customer_in_state

state
Arizona       1937
California    3552
Nevada         993
Oregon        2909
Washington     888
dtype: int64

In [95]:
marketing_customer_in_state= marketing_customer_in_state[marketing_customer_in_state > 500]

In [96]:
marketing_customer_in_state

state
Arizona       1937
California    3552
Nevada         993
Oregon        2909
Washington     888
dtype: int64

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [101]:
marketing_customer_gender_grouped = marketing_customer.groupby("gender")["customer_lifetime_value"].agg(['min','max','median'])
marketing_customer_gender_grouped

,min,max,median
gender,,,
F,1898.683686,73225.95652,5714.165863
M,1898.007675,83325.38119,5824.224192


In [102]:
marketing_customer_edu_grouped = marketing_customer.groupby("education")["customer_lifetime_value"].agg(['min','max','median'])
marketing_customer_edu_grouped

,min,max,median
education,,,
Bachelor,1898.007675,73225.95652,5579.380404
College,1898.683686,61850.18803,5813.629209
Doctor,2267.604038,44856.11397,5423.136755
High School or Below,1940.981221,83325.38119,6189.348131
Master,2272.307310,51016.06704,5680.467468


## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [141]:
marketing_customer["effective_to_date"] = pd.to_datetime(
    marketing_customer["effective_to_date"],
    format="%m/%d/%y"
)

In [142]:
marketing_customer["month"] = marketing_customer["effective_to_date"].dt.strftime("%b")

In [143]:
marketing_customer_pivot = marketing_customer.pivot_table(index="state", columns="month",values="number_of_policies",aggfunc="sum", fill_value=0)
marketing_customer_pivot

month,Feb,Jan
state,,
Arizona,2864,3052
California,4929,5673
Nevada,1278,1493
Oregon,3969,4697
Washington,1225,1358


6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [144]:
marketing_customer_pivot = marketing_customer.groupby(["state","month"])["number_of_policies"].sum()
marketing_customer_pivot

state       month
Arizona     Feb      2864
            Jan      3052
California  Feb      4929
            Jan      5673
Nevada      Feb      1278
            Jan      1493
Oregon      Feb      3969
            Jan      4697
Washington  Feb      1225
            Jan      1358
Name: number_of_policies, dtype: int64

In [145]:
marketing_customer_state = marketing_customer.groupby("state")["number_of_policies"].sum()
marketing_customer_state= marketing_customer_state.sort_values(ascending=False)

In [146]:
marketing_customer_state = marketing_customer_state[1:4]

In [147]:
marketing_customer_state

state
Oregon     8666
Arizona    5916
Nevada     2771
Name: number_of_policies, dtype: int64

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
# your code goes here